# Phase 1: Data Preparation and Exploration

**Goal:** Download the synthetic printed prescription dataset, explore images and metadata, split safely into dev (200) and test (300) images by image name, and prepare test sets for handwriting.

- Kaggle: `priyanshuyav13/synthetic-medical-prescription-ocr-dataset` (2,000 images)
- Roboflow: Handwritten prescription sample set
- Kaggle BD: `mamun1113/doctors-handwritten-prescription-bd-dataset`

## 0. Setup Cell (Repo Root & CUDA Check)
This cell locates the project root, adds `src/` to `sys.path`, and checks GPU availability.

In [ ]:
# --- SETUP CELL ---
import os, sys
from pathlib import Path

# Locate repo root by finding the 'src' directory
root = Path.cwd().resolve()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
if not (root / "src").exists():
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repo root:", root)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Imports and Data Helpers

In [ ]:
import importlib, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import src.data as D
importlib.reload(D)
from src.data import DATA_ROOT, build_image_index, open_image, gold_records

print("DATA_ROOT is set to:", DATA_ROOT)

## 2. Download Printed Synthetic Dataset
Dataset: `priyanshuyav13/synthetic-medical-prescription-ocr-dataset`.
Ensure `~/.kaggle/kaggle.json` exists with permissions 600.

In [ ]:
syn_dir = os.path.join(DATA_ROOT, "synthetic")
os.makedirs(syn_dir, exist_ok=True)

try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("Downloading synthetic prescription dataset...")
    api.dataset_download_files(
        "priyanshuyav13/synthetic-medical-prescription-ocr-dataset",
        path=syn_dir,
        unzip=True
    )
    print("Downloaded and unzipped successfully.")
except Exception as e:
    print("Kaggle download note:", e)
    print(f"If offline, ensure images and CSVs are extracted manually to {syn_dir}")

found_csvs = glob.glob(f"{syn_dir}/**/*.csv", recursive=True)
print("Found CSV files:", found_csvs)

## 3. Load, Map Columns, and Clean Labels

In [ ]:
labels_matches = [p for p in glob.glob(f"{syn_dir}/**/*.csv", recursive=True) if "label" in os.path.basename(p).lower()]
if labels_matches:
    labels_path = labels_matches[0]
    labels = pd.read_csv(labels_path)
    labels.columns = [c.strip().lower() for c in labels.columns]
    print("Raw columns:", labels.columns.tolist())
    display(labels.head())

    col_map = {}
    for c in labels.columns:
        if "file" in c or "image" in c:   col_map[c] = "filename"
        elif "medic" in c or "drug" in c: col_map[c] = "medicine"
        elif "dos" in c:                  col_map[c] = "dosage"
        elif "freq" in c:                 col_map[c] = "frequency"
    labels = labels.rename(columns=col_map)
    assert {"filename", "medicine", "dosage", "frequency"} <= set(labels.columns), f"Missing columns in: {labels.columns}"

    syn_index = build_image_index(syn_dir)
    labels["filename"] = labels["filename"].astype(str).map(os.path.basename)
    labels["has_image"] = labels["filename"].isin(syn_index)
    print("Total rows:", len(labels), "| Unique images:", labels["filename"].nunique(),
          "| Rows without an image file:", (~labels["has_image"]).sum())
    labels = labels[labels["has_image"]].copy()

    for c in ["medicine", "dosage", "frequency"]:
        labels[c] = labels[c].astype(str).str.strip()

    print("\nUnique medicines count:", labels["medicine"].nunique())
    print("\nTop 15 Medicines:")
    print(labels["medicine"].value_counts().head(15))
    print("\nTop 10 Dosages:")
    print(labels["dosage"].value_counts().head(10))
    print("\nTop 10 Frequencies:")
    print(labels["frequency"].value_counts().head(10))
else:
    print("No labels.csv found yet under", syn_dir)

## 4. Visual Inspection (First 6 Images & Metadata)

In [ ]:
if 'labels' in locals() and len(labels) > 0:
    fns = sorted(labels["filename"].unique())[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, fn in zip(axes.ravel(), fns):
        ax.imshow(open_image(fn, syn_index))
        ax.axis("off")
        ax.set_title(fn, fontsize=10)
    plt.tight_layout()
    plt.show()

meta_matches = [p for p in glob.glob(f"{syn_dir}/**/*.csv", recursive=True) if "meta" in os.path.basename(p).lower()]
if meta_matches:
    meta = pd.read_csv(meta_matches[0])
    print("Metadata preview:")
    display(meta.head())

## 5. Split by Image (200 Dev, 300 Test) and Save Gold CSVs
We split strictly on the image level with random seed 42 so medicines from the same prescription are never leaked between splits.

In [ ]:
if 'labels' in locals() and len(labels) > 0:
    files_all = sorted(labels["filename"].unique())
    rng = np.random.RandomState(42)
    rng.shuffle(files_all)
    dev_files = files_all[:200]
    test_files = files_all[200:500]

    cols = ["filename", "medicine", "dosage", "frequency"]
    os.makedirs("data", exist_ok=True)
    labels[labels["filename"].isin(dev_files)][cols].to_csv("data/dev_gold.csv", index=False)
    labels[labels["filename"].isin(test_files)][cols].to_csv("data/test_gold.csv", index=False)
    labels[["medicine"]].drop_duplicates().to_csv("data/synthetic_drugs.csv", index=False)
    print(f"Saved data/dev_gold.csv ({len(dev_files)} images)")
    print(f"Saved data/test_gold.csv ({len(test_files)} images)")
    print(f"Saved data/synthetic_drugs.csv ({labels['medicine'].nunique()} unique drugs)")

## 6. Doctor's Handwritten Prescription BD (Word-Level Test Crops)
Download dataset `mamun1113/doctors-handwritten-prescription-bd-dataset` and sample 100 test crops plus training vocabulary.

In [ ]:
bd_dir = os.path.join(DATA_ROOT, "bd_words")
os.makedirs(bd_dir, exist_ok=True)

try:
    api.dataset_download_files(
        "mamun1113/doctors-handwritten-prescription-bd-dataset",
        path=bd_dir,
        unzip=True
    )
    print("BD dataset downloaded.")
except Exception as e:
    print("BD download notice:", e)

bd_csvs = glob.glob(f"{bd_dir}/**/*.csv", recursive=True)
print("BD CSVs found:", bd_csvs)

bd_index = build_image_index(bd_dir)

def load_bd(csv_path):
    d = pd.read_csv(csv_path)
    d.columns = [c.strip().lower() for c in d.columns]
    fn = next(c for c in d.columns if "image" in c or "file" in c)
    lab = next((c for c in d.columns if "medicine" in c), None) or next(c for c in d.columns if "name" in c)
    d = d.rename(columns={fn: "filename", lab: "label"})
    d["label"] = d["label"].astype(str).str.strip()
    d["filename"] = d["filename"].astype(str).map(os.path.basename)
    return d[d["filename"].isin(bd_index)][["filename", "label"]]

train_candidates = [p for p in bd_csvs if "train" in p.lower()]
test_candidates = [p for p in bd_csvs if "test" in p.lower()]
if train_candidates and test_candidates:
    bd_train = load_bd(train_candidates[0])
    bd_test = load_bd(test_candidates[0])
    print("Loaded BD train:", len(bd_train), "test:", len(bd_test))
    os.makedirs("data", exist_ok=True)
    bd_test.sample(min(100, len(bd_test)), random_state=42).to_csv("data/bd_word_test.csv", index=False)
    pd.DataFrame({"label": sorted(set(bd_train["label"]))}).to_csv("data/bd_vocab.csv", index=False)
    print("Saved data/bd_word_test.csv and data/bd_vocab.csv")

## 7. RxHandBD Label Candidates (Zenodo)
If `train_labels.csv` and `test_labels.csv` from Zenodo RxHandBD are placed in `~/ocr_data/rxhandbd/`, generate candidate drug vocabulary.

In [ ]:
rx_dir = os.path.join(DATA_ROOT, "rxhandbd")
rx_train = os.path.join(rx_dir, "train_labels.csv")
rx_test = os.path.join(rx_dir, "test_labels.csv")

if os.path.exists(rx_train) and os.path.exists(rx_test):
    rx = pd.concat([pd.read_csv(rx_train), pd.read_csv(rx_test)])
    label_col = rx.columns[-1]
    cands = sorted({str(w).strip() for w in rx[label_col] if str(w).strip().isalpha() and len(str(w).strip()) >= 5})
    os.makedirs("data", exist_ok=True)
    pd.DataFrame({"candidate": cands}).to_csv("data/rxhandbd_candidates.csv", index=False)
    print(f"Extracted {len(cands)} candidate entries to data/rxhandbd_candidates.csv")
else:
    print(f"Optional: Place RxHandBD label files into {rx_dir} to extract additional candidate vocabulary.")